# 02_tune_surrogate: Step 2 - tune the XGBoost surrogate with PSO, GA and HHO (equal budget).

```
Fitness = 5-fold grouped (leave-days-out) CV RMSE on the training days.
Held-out test days are never seen during tuning. Outputs -> outputs/surrogate/.
Script asli: `scripts/02_tune_surrogate.py` [--budget 200] [--seeds 3] — parameter kini diatur di sel `args` di bawah.
```

Dikonversi dari `scripts/02_tune_surrogate.py`. Jalankan dari folder `notebooks/`.

In [1]:
from types import SimpleNamespace
import json
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve().parent   # notebook runs from AI-OBEP-MLPaper/notebooks
sys.path.insert(0, str(ROOT / "src"))
from obep_mlpaper.data import FEATURES, TARGET, day_split, hourly_dataset
from obep_mlpaper.metaheuristics import ALGORITHMS
from obep_mlpaper.surrogate import XGB_SPACE, make_xgb, metrics, xgb_cv_fitness

# Parameters (were command-line arguments in the script; edit here)
args = SimpleNamespace(budget=200, seeds=3, pop=15)

out = ROOT / "outputs" / "surrogate"
out.mkdir(parents=True, exist_ok=True)

df = hourly_dataset()
train, test = day_split(df)
X, y, g = df[FEATURES].to_numpy(), df[TARGET].to_numpy(), df["day"].to_numpy()
Xtr, ytr, gtr = X[train], y[train], g[train]
fitness = xgb_cv_fitness(Xtr, ytr, gtr)

In [2]:
runs, traces, logs = [], {}, []
for algo, fn in ALGORITHMS.items():
    for seed in range(args.seeds):
        t0 = time.time()
        kw = {"PSO": "n_particles", "GA": "pop_size", "HHO": "n_hawks"}[algo]
        r = fn(fitness, XGB_SPACE, budget=args.budget, seed=seed, **{kw: args.pop})
        m = make_xgb(r["best_params"]).fit(Xtr, ytr)
        te = metrics(y[test], np.clip(m.predict(X[test]), 0, None))
        runs.append(dict(algo=algo, seed=seed, cv_rmse=r["best_f"], n_evals=r["n_evals"],
                         minutes=(time.time() - t0) / 60, **{f"test_{k}": v for k, v in te.items()},
                         params=json.dumps(r["best_params"])))
        traces[f"{algo}_s{seed}"] = r["trace"]
        logs += [dict(algo=algo, seed=seed, **row) for row in r["log"]]
        print(f"{algo} seed={seed}: CV-RMSE={r['best_f']:.4f}  test R2={te['R2']:.4f} "
              f"({(time.time() - t0) / 60:.1f} min)", flush=True)

runs = pd.DataFrame(runs)
runs.to_csv(out / "metaheuristic_runs.csv", index=False)
pd.DataFrame(logs).to_csv(out / "metaheuristic_eval_log.csv", index=False)
L = max(len(t) for t in traces.values())
pd.DataFrame({k: np.pad(t, (0, L - len(t)), mode="edge") for k, t in traces.items()}).to_csv(
    out / "metaheuristic_convergence.csv", index_label="evaluation")

summary = runs.groupby("algo").agg(cv_rmse_mean=("cv_rmse", "mean"), cv_rmse_std=("cv_rmse", "std"),
                                   cv_rmse_best=("cv_rmse", "min"), test_R2_mean=("test_R2", "mean"),
                                   test_RMSE_mean=("test_RMSE", "mean"), minutes=("minutes", "mean"))
summary.to_csv(out / "metaheuristic_summary.csv")
print(summary.round(4))
best = runs.loc[runs.cv_rmse.idxmin()]
(out / "best_xgb_params.json").write_text(json.dumps(dict(
    algo=best.algo, seed=int(best.seed), cv_rmse=float(best.cv_rmse),
    params=json.loads(best.params), per_algo={a: json.loads(runs[runs.algo == a].sort_values("cv_rmse").iloc[0].params)
                                              for a in ALGORITHMS}), indent=2))
print("best:", best.algo, best.seed, best.params)

PSO seed=0: CV-RMSE=36.5411  test R2=0.9414 (3.1 min)


PSO seed=1: CV-RMSE=36.5911  test R2=0.9404 (2.5 min)


PSO seed=2: CV-RMSE=36.4876  test R2=0.9393 (1.3 min)


GA seed=0: CV-RMSE=38.0001  test R2=0.9394 (3.2 min)


GA seed=1: CV-RMSE=36.6079  test R2=0.9379 (2.5 min)


GA seed=2: CV-RMSE=37.1884  test R2=0.9384 (1.6 min)


HHO seed=0: CV-RMSE=38.5371  test R2=0.9331 (2.3 min)


HHO seed=1: CV-RMSE=36.7808  test R2=0.9376 (2.2 min)


HHO seed=2: CV-RMSE=37.6438  test R2=0.9380 (1.4 min)


      cv_rmse_mean  cv_rmse_std  cv_rmse_best  test_R2_mean  test_RMSE_mean  \
algo                                                                          
GA         37.2654       0.6993       36.6079        0.9386         36.8356   
HHO        37.6539       0.8782       36.7808        0.9362         37.5324   
PSO        36.5399       0.0518       36.4876        0.9404         36.2832   

      minutes  
algo           
GA     2.4318  
HHO    1.9967  
PSO    2.2963  
best: PSO 2 {"n_estimators": 126, "learning_rate": 0.030937243539859584, "max_depth": 5, "min_child_weight": 8.867723218012523, "subsample": 0.5699753277626409, "colsample_bytree": 1.0, "gamma": 4.606069605335564, "reg_lambda": 0.07719050189780147, "reg_alpha": 0.007836700939702651}
